In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from gtda.pipeline import Pipeline
from gtda.time_series import Resampler
from gtda.diagrams import PersistenceEntropy, Scaler, HeatKernel, BettiCurve
import numpy as np
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters, TakensEmbedding
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.metaestimators import CollectionTransformer
import plotly.express as px

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


In [ ]:
unstable_signals, unstable_df = read_files(DATA_3W_UNSTABLE, "P-TPT",3000)

In [ ]:
PatoBar = 1/100000
unstable_df = unstable_df.apply(lambda x: x*PatoBar) 
unstable_signals = unstable_signals * PatoBar
unstable_df

In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))
freq_units = 0.1/len(unstable_df[331])
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('instabilities BHP stats')
data[0].set_xlabel('time')
data[0].set_ylabel('BHP (Bar)')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP (Bar)')
data[2].set_ylabel('count')

data[1].set_yscale('log')
data[1].set_xscale('log')

ss_BHP = unstable_df[236]

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()

In [ ]:
# write file for making figures  
file = open("instability_3W.dat", "w")
for i in ss_BHP:
    tmp = str(i) + "\n"
    file.writelines(tmp)
file.close()

In [ ]:
max_time_delay = 55
max_embedding_dimension = 12
stride = 1

signal = ss_BHP 

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
embedding_dimension = optimal_time_delay
embedding_time_delay = optimal_time_delay

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_embedded_pca = pca.fit_transform(y_embedded)

norm = plot_point_cloud(y_embedded_pca)
norm

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_embedded[None,:,:])

In [ ]:
# write file for making figures  
file1 = open("Inst3W_Pers_H0.dat", "w")
file2 = open("Inst3W_Pers_H1.dat", "w")
for i in PerDiagram[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
batch_analyzer(unstable_df, 3, 20, 130)

In [ ]:
TE = TakensEmbedding(time_delay=125, dimension=8, stride=3)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PE = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)
Betti = BettiCurve()

unstable_point_cloud  = TE.fit_transform(unstable_signals)
unstable_diagrams = VRP.fit_transform(unstable_point_cloud)
unstable_entropy = PE.fit_transform(unstable_diagrams)
unstable_entropynorm = PE_norm.fit_transform(unstable_diagrams)
unstable_Betti = Betti.fit_transform(unstable_diagrams)

In [ ]:
tmp_entropies = unstable_entropy
Entropy_H0 = []
Entropy_H1 = []
tmp_normalized = unstable_entropynorm
Entropy_H0_norm = []
Entropy_H1_norm = []

for item in tmp_entropies:
    Entropy_H0.append(item[0])
    Entropy_H1.append(item[1])

for item in tmp_normalized:
    Entropy_H0_norm.append(item[0])
    Entropy_H1_norm.append(item[1])
    
Entropy_H1_series = pd.Series(Entropy_H1)
Entropy_H1_norm_series = pd.Series(Entropy_H1_norm)

entropies, ts = plt.subplots(2,figsize=(10,8),sharex = True)

entropies.suptitle('instabilities dataset persistent entropies')
ts[1].set_xlabel("# timeseries")
ts[0].set_ylabel('Entropy')
ts[1].set_ylabel('normalized  Entropy')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(Entropy_H1,'r-')
ts[0].plot(Entropy_H1_series.rolling(20).mean(), 'g-')
ts[0].plot(Entropy_H0,'b-')

ts[1].set_ylim([-20,40])
ts[1].plot(Entropy_H1_norm,'r-')
ts[1].plot(Entropy_H1_norm_series.rolling(20).mean(), 'g-')
ts[1].plot(Entropy_H0_norm,'b-')

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('unstable operations dataset persistent entropies')

data[0].set_xlabel('Entropy H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Entropy H$_1$")
data[1].set_ylabel('count')

data[0].set_xlim(8,10)
#data[1].set_xlim(0,8)
data[0].hist(Entropy_H0, bins=50)
data[1].hist(Entropy_H1, bins=25)

plt.tight_layout()

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('unstable operations dataset normalized persistent entropies')

data[0].set_xlabel('Normalized Entropy H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Normalized Entropy H$_1$")
data[1].set_ylabel('count')

data[0].set_xlim(0,7)
data[1].set_xlim(-5,8)
data[0].hist(Entropy_H0_norm, bins=25)
data[1].hist(Entropy_H1_norm, bins=100)

plt.tight_layout()

In [ ]:
import statistics as stats
meanBettiH0 = []
medianBettiH0 = []
modeBettiH0 = []

for i in range(len(unstable_Betti[:,0,0])):
    meanBettiH0.append(stats.mean(unstable_Betti[i,0,:]))
    medianBettiH0.append(stats.median(unstable_Betti[i,0,:]))
    modeBettiH0.append(stats.mode(unstable_Betti[i,0,:]))
    
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH0), name="Mean of Betti curve H_0")
figBetti.add_scatter(y=(medianBettiH0), name="Median of Betti curve H_0")
#figBetti.add_scatter(y=(modeBettiH0),name="Mode of Betti curve H_0")
figBetti.show()


In [ ]:
meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []

for i in range(len(unstable_Betti[:,1,0])):
    meanBettiH1.append(stats.mean(unstable_Betti[i,1,:]))
    medianBettiH1.append(stats.median(unstable_Betti[i,1,:]))
    modeBettiH1.append(stats.mode(unstable_Betti[i,1,:]))
    
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
figBetti.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
figBetti.add_scatter(y=(modeBettiH1),name="Mode of Betti curve H_1")
figBetti.show()

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))

stats_entropy.suptitle('unstable operations dataset normalized persistent entropies')

data[0].set_xlabel('Mean of Betti curve for H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Mean of Betti curve for H$_1$")
data[1].set_ylabel('count')

data[0].hist(meanBettiH0, bins=50)
data[1].hist(meanBettiH1)

plt.tight_layout()

In [ ]:
pers_H1 = []
pers_H0 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in unstable_diagrams: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence for unstable BHP')
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")

fig.show() 

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('Unstable operations dataset - Max Persistence')

data[0].set_xlabel('Mean of Betti Curve for H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Mean of Betti Curve for H$_1$")
data[1].set_ylabel('count')

data[0].hist(max_pers_H0, bins = 25)
data[1].hist(max_pers_H1, bins = 25)

plt.tight_layout()